# Generation-Based Code-Switching Evaluation (Llama-3.2-1B)

This notebook evaluates language steering using **actual text generation** and standard code-switching metrics (CSI, M-Index, I-Index, TLC) on Llama-3.2-1B.

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup: Load Model, Fit PCA, Inject Steering Layers

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from core.utils.device import DEVICE

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
model.to(DEVICE)
model.eval()
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

In [3]:
from core.gather_data.hidden_space import collect_hidden_space_by_language
from core.preprocess_data.flores_plus import load_flores_plus
from core.steering.pca import PCASteering
from core.integration.SteeredLlamaDecoderLayer import SteeredLlamaDecoderLayer

# Language pairs to evaluate
LANGUAGE_PAIRS = {
    "en-cn": {"flores_code": "cmn_Hans", "short": "cn", "steering_coeff": -5.00},
    "en-es": {"flores_code": "spa_Latn", "short": "es", "steering_coeff": -3.70},
    "en-ru": {"flores_code": "rus_Cyrl", "short": "ru", "steering_coeff": -3.50},
    "en-hin": {"flores_code": "hin_Deva", "short": "hin", "steering_coeff": -3.60},
}

MODEL_SHORT = "llama"
STEER_LAYERS = [14, 15]
N_SOURCE_TOKENS = 20

# Store original layers for restoration
original_layers = {}

def setup_steering_for_pair(model, tokenizer, pair_name, pair_info):
    """Fit pairwise PCA and inject steered decoder layers for one language pair."""
    flores_code = pair_info["flores_code"]
    short = pair_info["short"]

    train_df, _ = load_flores_plus(
        ["eng_Latn", flores_code],
        {"eng_Latn": "en", flores_code: short},
        train_size=200,
    )
    hidden_space, _ = collect_hidden_space_by_language(
        model, tokenizer, train_df, skip_first=True,
        cache_path=f"../../../.cache/hidden_space/{MODEL_SHORT}_{short}_flores_gen_pair.pt",
    )
    pca = PCASteering(
        cache_path=f"../../../.cache/pca/{MODEL_SHORT}_{short}_flores_gen_pair.pt"
    ).fit(hidden_space)

    replacement_layers = []
    for index in STEER_LAYERS:
        layer = SteeredLlamaDecoderLayer(
            model.config, index, pca, maintain_direction=True, n_source_tokens=N_SOURCE_TOKENS
        ).to(model.device)
        original_layers[index] = model.model.layers[index]
        layer.load_state_dict(original_layers[index].state_dict())
        model.model.layers[index] = layer
        replacement_layers.append(layer)

    return replacement_layers

def restore_original_layers(model):
    """Put back the original (unsteered) decoder layers."""
    for index, layer in original_layers.items():
        model.model.layers[index] = layer

print(f"Pairwise PCA helpers ready (layers {STEER_LAYERS}, maintain_direction=True, n_source_tokens={N_SOURCE_TOKENS})")

Pairwise PCA helpers ready (layers [14, 15], maintain_direction=True, n_source_tokens=20)


In [4]:
from core.evaluation.language_id import load_lid_model

lid_model = load_lid_model()
print("FastText LID model loaded")

FastText LID model loaded


## 2. Load Code-Switching Data and Generate Text

In [5]:
import json

import pandas as pd

# Load TED Talks code-switching data
data_path = "../../../data/ted_talks_code_switching_second_half.jsonl"
code_switching_data = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        code_switching_data.append(json.loads(line))

df = pd.DataFrame(code_switching_data)
print(f"Loaded {len(df)} TED Talks samples")
print(f"Columns: {list(df.columns)}")

Loaded 2467 TED Talks samples
Columns: ['eng_Latn', 'spa_Latn', 'cmn_Hans', 'rus_Cyrl', 'hin_Deva']


In [6]:
from pathlib import Path

import torch
from core.evaluation.generation import generate_continuation
from tqdm import tqdm

MAX_SAMPLES = 500  # Set to None for full dataset
MAX_NEW_TOKENS = 100

cache_path = Path("../../../.cache/generation/llama_generation_results_KL_coeff.pt")

if cache_path.exists():
    results_by_pair = torch.load(cache_path, weights_only=False)
    print(f"Loaded cached generation results from {cache_path}")
else:
    results_by_pair = {}

    for pair_name, pair_info in LANGUAGE_PAIRS.items():
        flores_code = pair_info["flores_code"]
        steering_coeff = pair_info["steering_coeff"]

        if flores_code not in df.columns:
            print(f"Skipping {pair_name}: column {flores_code} not in data")
            continue

        print(f"\n{'=' * 60}")
        print(f"Processing {pair_name} (steering coeff: {steering_coeff})")
        print(f"{'=' * 60}")

        # Fit pairwise PCA and inject steered layers for this pair
        replacement_layers = setup_steering_for_pair(model, tokenizer, pair_name, pair_info)

        n_samples = min(MAX_SAMPLES, len(df)) if MAX_SAMPLES else len(df)
        pair_results = []

        for idx in tqdm(range(n_samples), desc=pair_name):
            eng_text = df.iloc[idx]["eng_Latn"]
            mixed_text = df.iloc[idx][flores_code]

            # Condition A: English baseline (no steering)
            for layer in replacement_layers:
                layer.disable()
            gen_eng, ids_eng = generate_continuation(model, tokenizer, eng_text, max_new_tokens=MAX_NEW_TOKENS)

            # Condition B: Code-switched input, no steering
            for layer in replacement_layers:
                layer.disable()
            gen_unsteered, ids_unsteered = generate_continuation(
                model, tokenizer, mixed_text, max_new_tokens=MAX_NEW_TOKENS
            )

            # Condition C: Code-switched input, with steering
            if steering_coeff != 0:
                for layer in replacement_layers:
                    layer.reset()
                    layer.set_steering_direction(steering_coeff)
                gen_steered, ids_steered = generate_continuation(
                    model, tokenizer, mixed_text, max_new_tokens=MAX_NEW_TOKENS
                )
                for layer in replacement_layers:
                    layer.disable()
            else:
                # No steering for this pair (e.g., Hindi)
                gen_steered, ids_steered = gen_unsteered, ids_unsteered

            pair_results.append(
                {
                    "idx": idx,
                    "eng_text": eng_text,
                    "mixed_text": mixed_text,
                    "gen_eng": gen_eng,
                    "gen_unsteered": gen_unsteered,
                    "gen_steered": gen_steered,
                    "ids_eng": ids_eng,
                    "ids_unsteered": ids_unsteered,
                    "ids_steered": ids_steered,
                }
            )

        results_by_pair[pair_name] = pair_results
        print(f"Generated {len(pair_results)} samples for {pair_name}")

        # Restore original layers before next pair
        restore_original_layers(model)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(results_by_pair, cache_path)
    print(f"Saved generation results to {cache_path}")


Processing en-cn (steering coeff: -5.0)


en-cn:   0%|                                                                             | 0/500 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


en-hin: 100%|██████████████████████████████████████████████████████████████████| 500/500 [37:57<00:00,  4.54s/it]

en-hin: 100%|██████████████████████████████████████████████████████████████████| 500/500 [37:57<00:00,  4.55s/it]

Generated 500 samples for en-hin


Saved generation results to ../../../.cache/generation/llama_generation_results_KL_coeff.pt


## 3. Token-Level Language Classification

In [7]:
from core.evaluation.language_id import classify_generated_text

classified_results_by_pair = {}

for pair_name, pair_results in results_by_pair.items():
    print(f"\nClassifying tokens for {pair_name}...")
    classified = []

    for result in tqdm(pair_results, desc=f"Classifying {pair_name}"):
        labels_eng = classify_generated_text(result["gen_eng"], tokenizer, lid_model)
        labels_unsteered = classify_generated_text(result["gen_unsteered"], tokenizer, lid_model)
        labels_steered = classify_generated_text(result["gen_steered"], tokenizer, lid_model)

        classified.append(
            {
                **result,
                "labels_eng": [lab for _, lab in labels_eng],
                "labels_unsteered": [lab for _, lab in labels_unsteered],
                "labels_steered": [lab for _, lab in labels_steered],
                "tokens_eng": [tok for tok, _ in labels_eng],
                "tokens_unsteered": [tok for tok, _ in labels_unsteered],
                "tokens_steered": [tok for tok, _ in labels_steered],
            }
        )

    classified_results_by_pair[pair_name] = classified
    print(f"Classified {len(classified)} samples for {pair_name}")


Classifying tokens for en-cn...


Classifying en-cn:   0%|                                                                 | 0/500 [00:00<?, ?it/s]

Classifying en-cn:   2%|█▏                                                     | 11/500 [00:00<00:04, 107.64it/s]

Classifying en-cn:   6%|███▏                                                   | 29/500 [00:00<00:03, 149.04it/s]

Classifying en-cn:   9%|█████▏                                                 | 47/500 [00:00<00:02, 159.61it/s]

...

Classifying en-hin:  90%|███████████████████████████████████████████████▊     | 451/500 [00:02<00:00, 178.95it/s]

Classifying en-hin:  94%|█████████████████████████████████████████████████▋   | 469/500 [00:02<00:00, 179.16it/s]

Classifying en-hin:  98%|███████████████████████████████████████████████████▋ | 488/500 [00:02<00:00, 180.54it/s]

Classifying en-hin: 100%|█████████████████████████████████████████████████████| 500/500 [00:02<00:00, 179.88it/s]

Classified 500 samples for en-hin


## 4. Compute Code-Switching Metrics

In [8]:
import numpy as np

from core.evaluation.code_switching_metrics import compute_all_metrics

TARGET_LANG = "en"

metrics_by_pair = {}

for pair_name, classified in classified_results_by_pair.items():
    print(f"\nComputing metrics for {pair_name}...")

    pair_metrics = {"eng": [], "unsteered": [], "steered": []}

    for result in classified:
        pair_metrics["eng"].append(compute_all_metrics(result["labels_eng"], TARGET_LANG))
        pair_metrics["unsteered"].append(compute_all_metrics(result["labels_unsteered"], TARGET_LANG))
        pair_metrics["steered"].append(compute_all_metrics(result["labels_steered"], TARGET_LANG))

    metrics_by_pair[pair_name] = pair_metrics

print("\nDone computing all metrics.")


Computing metrics for en-cn...

Computing metrics for en-es...

Computing metrics for en-ru...

Computing metrics for en-hin...



Done computing all metrics.


## 5. Results Tables

In [9]:
# Table A: CSI comparison across all language pairs
print("=" * 90)
print("Table A: Code-Switching Index (CSI) — fraction of non-English tokens (lower = better)")
print("=" * 90)
print(f"{'Language Pair':<18} {'CSI (English)':<15} {'CSI (Unsteered)':<18} {'CSI (Steered)':<16} {'Delta CSI':<12}")
print("-" * 90)

for pair_name, metrics in metrics_by_pair.items():
    csi_eng = np.mean([m["csi"] for m in metrics["eng"]])
    csi_unst = np.mean([m["csi"] for m in metrics["unsteered"]])
    csi_st = np.mean([m["csi"] for m in metrics["steered"]])
    delta = csi_st - csi_unst
    print(f"{pair_name:<18} {csi_eng:<15.4f} {csi_unst:<18.4f} {csi_st:<16.4f} {delta:<12.4f}")

print()

Table A: Code-Switching Index (CSI) — fraction of non-English tokens (lower = better)
Language Pair      CSI (English)   CSI (Unsteered)    CSI (Steered)    Delta CSI   
------------------------------------------------------------------------------------------
en-cn              0.0177          0.6353             0.0329           -0.6024     
en-es              0.0177          0.6240             0.2306           -0.3935     
en-ru              0.0177          0.5923             0.0236           -0.5687     
en-hin             0.0177          0.8067             0.0437           -0.7631     



In [10]:
# Table B: All metrics per language pair
METRIC_NAMES = ["csi", "m_index", "i_index", "mean_span_length", "tlc"]
METRIC_LABELS = ["CSI", "M-Index", "I-Index", "Mean Span Length", "TLC"]

for pair_name, metrics in metrics_by_pair.items():
    print(f"\n{'=' * 80}")
    print(f"Table B: All Metrics for {pair_name}")
    print(f"{'=' * 80}")
    print(f"{'Metric':<22} {'English':<12} {'Unsteered':<12} {'Steered':<12} {'Improvement':<14}")
    print("-" * 80)

    for metric_key, metric_label in zip(METRIC_NAMES, METRIC_LABELS):
        val_eng = np.mean([m[metric_key] for m in metrics["eng"]])
        val_unst = np.mean([m[metric_key] for m in metrics["unsteered"]])
        val_st = np.mean([m[metric_key] for m in metrics["steered"]])

        if metric_key in ["tlc", "mean_span_length"]:
            # Higher is better
            if val_unst != 0:
                improvement = f"+{((val_st - val_unst) / val_unst) * 100:.1f}%"
            else:
                improvement = "N/A"
        else:
            # Lower is better
            if val_unst != 0:
                improvement = f"{((val_st - val_unst) / val_unst) * 100:.1f}%"
            else:
                improvement = "N/A"

        print(f"{metric_label:<22} {val_eng:<12.4f} {val_unst:<12.4f} {val_st:<12.4f} {improvement:<14}")


Table B: All Metrics for en-cn
Metric                 English      Unsteered    Steered      Improvement   
--------------------------------------------------------------------------------
CSI                    0.0177       0.6353       0.0329       -94.8%        
M-Index                0.0236       0.1888       0.0374       -80.2%        
I-Index                0.0204       0.0937       0.0137       -85.4%        
Mean Span Length       80.6175      35.2591      64.6077      +83.2%        
TLC                    0.9823       0.3647       0.9671       +165.2%       

Table B: All Metrics for en-es
Metric                 English      Unsteered    Steered      Improvement   
--------------------------------------------------------------------------------
CSI                    0.0177       0.6240       0.2306       -63.1%        
M-Index                0.0236       0.1724       0.1198       -30.5%        
I-Index                0.0204       0.1075       0.1310       21.8%         
Mean

In [11]:
# Table C: Qualitative examples
N_EXAMPLES = 5

for pair_name, classified in classified_results_by_pair.items():
    print(f"\n{'=' * 80}")
    print(f"Table C: Qualitative Examples for {pair_name}")
    print(f"{'=' * 80}")

    for i in range(min(N_EXAMPLES, len(classified))):
        result = classified[i]
        print(f"\n--- Sample {i} ---")
        print(f"Input (code-switched): {result['mixed_text'][:100]}...")
        print(f"\nGenerated (unsteered): {result['gen_unsteered'][:200]}")
        print(f"Token labels: {result['labels_unsteered'][:30]}")
        print(f"\nGenerated (steered):   {result['gen_steered'][:200]}")
        print(f"Token labels: {result['labels_steered'][:30]}")


Table C: Qualitative Examples for en-cn

--- Sample 0 ---
Input (code-switched): Good morning. How are you?(Laughter)It's been great, hasn't it? I've 被这整个事情震惊到了。事实上，我...

Generated (unsteered): 已经在这儿工作了五年了。I've been here for five years. (Laughter)I'm not sure how long I'll be here. (Laughter)I'm not sure how long I'll be here. (Laughter)I'm not sure how long I'll be here. (Laughter)I'm not s
Token labels: ['zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en']

Generated (steered):   在 and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, and the, 
Token labels: ['zh', 'zh', 'zh', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', '

## 6. Statistical Significance

In [12]:
from scipy import stats

print("=" * 80)
print("Statistical Significance Tests (Wilcoxon signed-rank test)")
print("H0: No difference between steered and unsteered CSI")
print("=" * 80)

for pair_name, metrics in metrics_by_pair.items():
    csi_unsteered = [m["csi"] for m in metrics["unsteered"]]
    csi_steered = [m["csi"] for m in metrics["steered"]]

    # Wilcoxon signed-rank test (paired, non-parametric)
    try:
        stat, p_value = stats.wilcoxon(csi_unsteered, csi_steered, alternative="greater")
        significant = "Yes" if p_value < 0.05 else "No"
    except ValueError:
        # All differences are zero (e.g., Hindi with coeff=0)
        stat, p_value, significant = 0, 1.0, "N/A (identical)"

    mean_unst = np.mean(csi_unsteered)
    mean_st = np.mean(csi_steered)
    ci_diff = 1.96 * np.std(np.array(csi_unsteered) - np.array(csi_steered)) / np.sqrt(len(csi_unsteered))

    print(f"\n{pair_name}:")
    print(f"  Mean CSI (unsteered): {mean_unst:.4f}")
    print(f"  Mean CSI (steered):   {mean_st:.4f}")
    print(f"  Mean difference:      {mean_unst - mean_st:.4f} +/- {ci_diff:.4f}")
    print(f"  Wilcoxon statistic:   {stat:.2f}")
    print(f"  p-value:              {p_value:.6f}")
    print(f"  Significant (p<0.05): {significant}")

Statistical Significance Tests (Wilcoxon signed-rank test)
H0: No difference between steered and unsteered CSI

en-cn:
  Mean CSI (unsteered): 0.6353
  Mean CSI (steered):   0.0329
  Mean difference:      0.6024 +/- 0.0353
  Wilcoxon statistic:   115284.50
  p-value:              0.000000
  Significant (p<0.05): Yes

en-es:
  Mean CSI (unsteered): 0.6240
  Mean CSI (steered):   0.2306
  Mean difference:      0.3935 +/- 0.0437
  Wilcoxon statistic:   93918.50
  p-value:              0.000000
  Significant (p<0.05): Yes

en-ru:
  Mean CSI (unsteered): 0.5923
  Mean CSI (steered):   0.0236
  Mean difference:      0.5687 +/- 0.0373
  Wilcoxon statistic:   107170.00
  p-value:              0.000000
  Significant (p<0.05): Yes

en-hin:
  Mean CSI (unsteered): 0.8067
  Mean CSI (steered):   0.0437
  Mean difference:      0.7631 +/- 0.0339
  Wilcoxon statistic:   108814.00
  p-value:              0.000000
  Significant (p<0.05): Yes
